# Google Routes API — Area Traffic Mapping

This notebook maps traffic conditions between a configurable focal location and multiple area entry points using the Google Maps Platform Routes API.

> **Before running this notebook:** replace all values marked `REPLACE_ME` in the configuration cell with your own study-area information.

One complete observation cycle uses one API request per origin–destination pair. Alternative routes can also be requested to improve road-network coverage.

## Time options

- `LIVE` — uses traffic conditions at the time of the API request.
- `SCHEDULED` — uses a specific future local date and time.

For Google Routes API `DRIVE` requests, a past `departureTime` is not supported. A scheduled time therefore must be in the future.

Outputs are saved locally in the configured output directory.


In [ ]:
%pip install -q requests pandas matplotlib


In [ ]:
import json
import math
import os
from datetime import datetime, timedelta, timezone
from getpass import getpass
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display

# -------------------------------------------------------------------
# STUDY AREA — MUST BE REPLACED BEFORE USE
# -------------------------------------------------------------------
# Replace every None / REPLACE_ME value below with your own study area.
# Do not publish confidential, restricted, or personally identifying
# coordinates in a public repository.

FOCAL_LOCATION = {
    "name": "REPLACE_ME_FOCAL_LOCATION_NAME",
    "latitude": None,   # REPLACE_ME: decimal latitude
    "longitude": None,  # REPLACE_ME: decimal longitude
}

AREA_ENTRY_POINTS = [
    {"name": "REPLACE_ME_ENTRY_01", "latitude": None, "longitude": None},
    {"name": "REPLACE_ME_ENTRY_02", "latitude": None, "longitude": None},
    {"name": "REPLACE_ME_ENTRY_03", "latitude": None, "longitude": None},
    {"name": "REPLACE_ME_ENTRY_04", "latitude": None, "longitude": None},
    {"name": "REPLACE_ME_ENTRY_05", "latitude": None, "longitude": None},
    {"name": "REPLACE_ME_ENTRY_06", "latitude": None, "longitude": None},
    {"name": "REPLACE_ME_ENTRY_07", "latitude": None, "longitude": None},
    {"name": "REPLACE_ME_ENTRY_08", "latitude": None, "longitude": None},
]

# REPLACE_ME if your study area uses another local timezone.
LOCAL_TIMEZONE = "Asia/Makassar"

BIDIRECTIONAL = True
COMPUTE_ALTERNATIVE_ROUTES = True
ROUTING_PREFERENCE = "TRAFFIC_AWARE_OPTIMAL"
POLYLINE_QUALITY = "HIGH_QUALITY"
TRAFFIC_MODEL = "BEST_GUESS"

# Local safeguard only. This is NOT a Google Cloud billing/quota control.
LOCAL_MONTHLY_SAFETY_LIMIT = 100
TIMEOUT_SECONDS = 60
OUTPUT_DIR = Path("output_google_routes_traffic")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def validate_study_area():
    """Stop execution until all public-template placeholders are replaced."""
    locations = [FOCAL_LOCATION, *AREA_ENTRY_POINTS]
    for idx, location in enumerate(locations):
        if str(location.get("name", "")).startswith("REPLACE_ME"):
            raise ValueError(
                f"Location name at index {idx} still contains REPLACE_ME. "
                "Replace it with your own value before running the notebook."
            )
        if location.get("latitude") is None or location.get("longitude") is None:
            raise ValueError(
                f"Coordinates for {location.get('name', idx)!r} are missing. "
                "Replace all None coordinate values before running the notebook."
            )

validate_study_area()

# -------------------------------------------------------------------
# DATE/TIME INPUT
# -------------------------------------------------------------------
print("TIME MODE")
print("  LIVE      = traffic at request time")
print("  SCHEDULED = predicted traffic for a specific FUTURE local date/time")
print()

TIME_SELECTION = input("Enter LIVE or SCHEDULED [LIVE]: ").strip().upper()
if not TIME_SELECTION:
    TIME_SELECTION = "LIVE"

if TIME_SELECTION not in {"LIVE", "SCHEDULED"}:
    raise ValueError("Please enter only LIVE or SCHEDULED.")

if TIME_SELECTION == "SCHEDULED":
    LOCAL_DATE = input("Enter local date (YYYY-MM-DD): ").strip()
    LOCAL_TIME = input("Enter local time (HH:MM, 24-hour format): ").strip()
    DEPARTURE_TIME_LOCAL = f"{LOCAL_DATE} {LOCAL_TIME}"
else:
    DEPARTURE_TIME_LOCAL = None

# -------------------------------------------------------------------
# GOOGLE MAPS PLATFORM API KEY
# -------------------------------------------------------------------
# Recommended for GitHub/public repositories:
#   export GOOGLE_MAPS_API_KEY="YOUR_PRIVATE_KEY"
# Never hard-code a real key in this notebook or commit it to Git.
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
if not API_KEY:
    API_KEY = getpass("Enter Google Maps API key (hidden input): ").strip()

if not API_KEY:
    raise RuntimeError("Google Maps API key was not provided.")

print()
print("Working directory :", Path.cwd().resolve())
print("Output folder     :", OUTPUT_DIR.resolve())
print("API key accepted and hidden.")


In [ ]:
ROUTES_URL = "https://routes.googleapis.com/directions/v2:computeRoutes"
COUNTER_FILE = OUTPUT_DIR / "_local_request_counter.json"
LOCAL_ZONE = ZoneInfo(LOCAL_TIMEZONE)

def waypoint(location):
    return {
        "location": {
            "latLng": {
                "latitude": float(location["latitude"]),
                "longitude": float(location["longitude"]),
            }
        }
    }

def parse_duration(value):
    if not value or not value.endswith("s"):
        raise ValueError(f"Unrecognized duration format: {value!r}")
    return float(value[:-1])

def haversine_km(point_1, point_2):
    lon_1, lat_1 = point_1
    lon_2, lat_2 = point_2
    radius_km = 6371.0088
    phi_1, phi_2 = math.radians(lat_1), math.radians(lat_2)
    delta_phi = phi_2 - phi_1
    delta_lambda = math.radians(lon_2 - lon_1)

    a = (
        math.sin(delta_phi / 2) ** 2
        + math.cos(phi_1) * math.cos(phi_2)
        * math.sin(delta_lambda / 2) ** 2
    )
    return 2 * radius_km * math.asin(math.sqrt(a))

def line_length_km(coordinates):
    return sum(
        haversine_km(coordinates[i], coordinates[i + 1])
        for i in range(len(coordinates) - 1)
    )

def read_local_counter():
    month_key = datetime.now(timezone.utc).strftime("%Y-%m")
    default = {"month_utc": month_key, "successful_requests": 0}

    if not COUNTER_FILE.exists():
        return default

    try:
        data = json.loads(COUNTER_FILE.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return default

    return data if data.get("month_utc") == month_key else default

def remaining_local_requests():
    used = int(read_local_counter().get("successful_requests", 0))
    return LOCAL_MONTHLY_SAFETY_LIMIT - used

def check_local_limit(required_requests=1):
    remaining = remaining_local_requests()
    if required_requests > remaining:
        raise RuntimeError(
            f"{required_requests} requests are required, but only "
            f"{remaining} remain under the local safety limit."
        )

def record_successful_request():
    counter = read_local_counter()
    counter["successful_requests"] = int(
        counter.get("successful_requests", 0)
    ) + 1
    counter["last_success_utc"] = datetime.now(timezone.utc).isoformat()
    COUNTER_FILE.write_text(
        json.dumps(counter, indent=2),
        encoding="utf-8"
    )

def prepare_departure_time_local(value):
    """Convert a local time YYYY-MM-DD HH:MM value to RFC3339 UTC for Routes API.

    LIVE mode returns no explicit departureTime so the API uses request time.
    DRIVE requests cannot specify a departureTime in the past.
    """
    now_utc = datetime.now(timezone.utc)

    if value is None or not str(value).strip():
        now_local = now_utc.astimezone(LOCAL_ZONE)
        return "LIVE", now_local, None

    try:
        local_naive = datetime.strptime(
            str(value).strip(),
            "%Y-%m-%d %H:%M"
        )
    except ValueError as exc:
        raise ValueError(
            "Date/time must use YYYY-MM-DD HH:MM, for example "
            "'2026-10-01 07:00'."
        ) from exc

    departure_local = local_naive.replace(tzinfo=LOCAL_ZONE)
    departure_utc = departure_local.astimezone(timezone.utc)

    if departure_utc < now_utc - timedelta(seconds=5):
        raise ValueError(
            "The selected local date/time is already in the past. "
            "Google Routes API does not support past departureTime values "
            "for DRIVE mode. Choose LIVE or enter a future date/time."
        )

    api_timestamp = departure_utc.isoformat(
        timespec="seconds"
    ).replace("+00:00", "Z")

    return "SCHEDULED", departure_local, api_timestamp

def request_traffic(origin, destination, departure_time_utc=None):
    check_local_limit(1)

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": (
            "routes.routeLabels,"
            "routes.distanceMeters,"
            "routes.duration,"
            "routes.staticDuration,"
            "routes.polyline,"
            "routes.travelAdvisory.speedReadingIntervals"
        ),
    }

    payload = {
        "origin": waypoint(origin),
        "destination": waypoint(destination),
        "travelMode": "DRIVE",
        "routingPreference": ROUTING_PREFERENCE,
        "computeAlternativeRoutes": COMPUTE_ALTERNATIVE_ROUTES,
        "extraComputations": ["TRAFFIC_ON_POLYLINE"],
        "polylineQuality": POLYLINE_QUALITY,
        "polylineEncoding": "GEO_JSON_LINESTRING",
    }

    # In LIVE mode departureTime is omitted, so Google uses request time.
    # In SCHEDULED mode, a future RFC3339 timestamp is supplied.
    if departure_time_utc is not None:
        payload["departureTime"] = departure_time_utc
        payload["trafficModel"] = TRAFFIC_MODEL

    response = requests.post(
        ROUTES_URL,
        headers=headers,
        json=payload,
        timeout=TIMEOUT_SECONDS,
    )

    if not response.ok:
        try:
            message = response.json().get(
                "error", {}
            ).get("message", response.text)
        except ValueError:
            message = response.text

        raise RuntimeError(
            f"Routes API request failed ({response.status_code}): {message}"
        )

    result = response.json()

    if not result.get("routes"):
        raise RuntimeError("Routes API returned no routes.")

    record_successful_request()
    return result


In [ ]:
time_mode, requested_departure_local, api_departure_time_utc = (
    prepare_departure_time_local(DEPARTURE_TIME_LOCAL)
)

requested_departure_local_text = requested_departure_local.isoformat(
    timespec="seconds"
)

print("Time mode              :", time_mode)
print("Local time              :", requested_departure_local_text)
print(
    "departureTime for API :",
    api_departure_time_utc or "automatic = request time"
)

od_plan = []

for entry in AREA_ENTRY_POINTS:
    od_plan.append({
        "direction": "FROM_CENTER",
        "origin": FOCAL_LOCATION,
        "destination": entry,
    })

    if BIDIRECTIONAL:
        od_plan.append({
            "direction": "TO_CENTER",
            "origin": entry,
            "destination": FOCAL_LOCATION,
        })

plan_df = pd.DataFrame([
    {
        "request_no": i,
        "direction": item["direction"],
        "origin": item["origin"]["name"],
        "destination": item["destination"]["name"],
    }
    for i, item in enumerate(od_plan, start=1)
])

required_requests = len(od_plan)
remaining_before = remaining_local_requests()
check_local_limit(required_requests)

print()
print(f"Requests in one observation cycle : {required_requests}")
print(f"Local requests remaining before   : {remaining_before}")
print(
    f"Remaining if all requests succeed : "
    f"{remaining_before - required_requests}"
)

display(plan_df)


In [ ]:
confirmation = input(
    f"Type RUN to send {len(od_plan)} API requests: "
).strip().upper()

if confirmation != "RUN":
    raise RuntimeError(
        "Traffic retrieval cancelled; no API requests were sent."
    )

captured_at_utc = datetime.now(timezone.utc)
captured_at_local = captured_at_utc.astimezone(LOCAL_ZONE)

area_responses = []
error_rows = []

for request_no, item in enumerate(od_plan, start=1):
    origin_name = item["origin"]["name"]
    destination_name = item["destination"]["name"]

    print(
        f"[{request_no:02d}/{len(od_plan):02d}] "
        f"{origin_name} -> {destination_name}"
    )

    try:
        accessed_at_utc = datetime.now(timezone.utc)
        accessed_at_local = accessed_at_utc.astimezone(LOCAL_ZONE)

        result = request_traffic(
            item["origin"],
            item["destination"],
            api_departure_time_utc,
        )

        area_responses.append({
            "request_no": request_no,
            "direction": item["direction"],
            "origin_name": origin_name,
            "destination_name": destination_name,
            "accessed_at_utc": accessed_at_utc.isoformat(),
            "accessed_at_local": accessed_at_local.isoformat(),
            "result": result,
        })

    except Exception as exc:
        error_rows.append({
            "request_no": request_no,
            "origin_name": origin_name,
            "destination_name": destination_name,
            "error": str(exc),
        })
        print("  FAILED:", exc)

print()
print(f"Successful requests : {len(area_responses)}")
print(f"Failed requests     : {len(error_rows)}")
print(f"Local limit left    : {remaining_local_requests()}")


In [ ]:
area_features = []
route_rows = []
segment_rows = []

for response_item in area_responses:
    for route_index, route in enumerate(
        response_item["result"]["routes"],
        start=1
    ):
        coordinates = route["polyline"]["geoJsonLinestring"]["coordinates"]
        intervals = route.get(
            "travelAdvisory", {}
        ).get("speedReadingIntervals", [])

        labels = route.get("routeLabels", [])

        route_type = (
            "ALTERNATIVE"
            if "DEFAULT_ROUTE_ALTERNATE" in labels
            else "PRIMARY"
        )

        duration_min = parse_duration(route["duration"]) / 60
        static_min = parse_duration(route["staticDuration"]) / 60

        route_id = (
            f"Q{response_item['request_no']:02d}_R{route_index:02d}"
        )

        route_rows.append({
            "route_id": route_id,
            "request_no": response_item["request_no"],
            "direction": response_item["direction"],
            "origin": response_item["origin_name"],
            "destination": response_item["destination_name"],
            "route_type": route_type,
            "distance_km": round(
                route["distanceMeters"] / 1000, 3
            ),
            "duration_with_traffic_min": round(duration_min, 2),
            "static_duration_min": round(static_min, 2),
            "traffic_delay_min": round(
                max(0, duration_min - static_min), 2
            ),
            "captured_at_utc": captured_at_utc.isoformat(),
            "captured_at_local": captured_at_local.isoformat(),
            "accessed_at_utc": response_item["accessed_at_utc"],
            "accessed_at_local": response_item["accessed_at_local"],
            "time_mode": time_mode,
            "requested_departure_local": requested_departure_local_text,
            "api_departure_time_utc": api_departure_time_utc,
        })

        for segment_no, interval in enumerate(intervals, start=1):
            start_index = int(
                interval.get("startPolylinePointIndex", 0)
            )
            end_index = int(
                interval["endPolylinePointIndex"]
            )

            status = interval.get("speed", "UNKNOWN")

            segment_coordinates = coordinates[
                start_index:min(end_index + 1, len(coordinates))
            ]

            if len(segment_coordinates) < 2:
                continue

            properties = {
                "route_id": route_id,
                "segment_no": segment_no,
                "request_no": response_item["request_no"],
                "direction": response_item["direction"],
                "origin": response_item["origin_name"],
                "destination": response_item["destination_name"],
                "route_type": route_type,
                "traffic_status": status,
                "length_km": round(
                    line_length_km(segment_coordinates), 4
                ),
                "captured_at_utc": captured_at_utc.isoformat(),
                "captured_at_local": captured_at_local.isoformat(),
                "accessed_at_utc": response_item["accessed_at_utc"],
                "accessed_at_local": response_item["accessed_at_local"],
                "time_mode": time_mode,
                "requested_departure_local": requested_departure_local_text,
                "api_departure_time_utc": api_departure_time_utc,
                "source": "Google Maps Platform Routes API",
            }

            segment_rows.append(properties.copy())

            area_features.append({
                "type": "Feature",
                "properties": properties,
                "geometry": {
                    "type": "LineString",
                    "coordinates": segment_coordinates,
                },
            })

route_df = pd.DataFrame(route_rows)
segment_df = pd.DataFrame(segment_rows)

print(f"Primary + alternative routes : {len(route_df)}")
print(f"Traffic segments             : {len(segment_df)}")

display(route_df.head(20))


In [ ]:
# Use the requested departure time in the filename for SCHEDULED mode.
# LIVE mode uses the actual capture time.
if time_mode == "SCHEDULED":
    output_time_id = requested_departure_local.strftime(
        "%Y%m%dT%H%M%S_LOCAL"
    )
    output_suffix = f"{output_time_id}_SCHEDULED"
else:
    output_time_id = captured_at_local.strftime(
        "%Y%m%dT%H%M%S_LOCAL"
    )
    output_suffix = f"{output_time_id}_LIVE"

route_csv_path = (
    OUTPUT_DIR / f"area_route_summary_{output_suffix}.csv"
)
segment_csv_path = (
    OUTPUT_DIR / f"area_traffic_segments_{output_suffix}.csv"
)
geojson_path = (
    OUTPUT_DIR / f"area_traffic_segments_{output_suffix}.geojson"
)
error_csv_path = (
    OUTPUT_DIR / f"area_request_errors_{output_suffix}.csv"
)
figure_path = (
    OUTPUT_DIR / f"area_traffic_map_{output_suffix}.png"
)

route_df.to_csv(
    route_csv_path,
    index=False,
    encoding="utf-8-sig"
)

segment_df.to_csv(
    segment_csv_path,
    index=False,
    encoding="utf-8-sig"
)

if error_rows:
    pd.DataFrame(error_rows).to_csv(
        error_csv_path,
        index=False,
        encoding="utf-8-sig"
    )

geojson = {
    "type": "FeatureCollection",
    "properties": {
        "focal_location": FOCAL_LOCATION["name"],
        "captured_at_utc": captured_at_utc.isoformat(),
        "captured_at_local": captured_at_local.isoformat(),
        "time_mode": time_mode,
        "requested_departure_local": requested_departure_local_text,
        "api_departure_time_utc": api_departure_time_utc,
        "traffic_model": (
            TRAFFIC_MODEL if time_mode == "SCHEDULED"
            else "LIVE/default request-time traffic"
        ),
        "successful_api_requests": len(area_responses),
        "source": "Google Maps Platform Routes API",
        "cache_notice": (
            "Review Google Maps Platform storage and attribution terms."
        ),
    },
    "features": area_features,
}

geojson_path.write_text(
    json.dumps(
        geojson,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

status_colors = {
    "NORMAL": "#2ca25f",
    "SLOW": "#fdae6b",
    "TRAFFIC_JAM": "#de2d26",
    "UNKNOWN": "#756bb1",
}

status_order = {
    "NORMAL": 0,
    "SLOW": 1,
    "TRAFFIC_JAM": 2,
    "UNKNOWN": 3,
}

ordered_features = sorted(
    area_features,
    key=lambda f: status_order.get(
        f["properties"]["traffic_status"], 3
    )
)

fig, ax = plt.subplots(figsize=(10, 10))
labels_used = set()

for feature in ordered_features:
    status = feature["properties"]["traffic_status"]
    coords = feature["geometry"]["coordinates"]

    label = status if status not in labels_used else None

    ax.plot(
        [p[0] for p in coords],
        [p[1] for p in coords],
        color=status_colors.get(
            status,
            status_colors["UNKNOWN"]
        ),
        linewidth=3.5,
        alpha=0.90,
        solid_capstyle="round",
        label=label,
    )

    labels_used.add(status)

ax.scatter(
    FOCAL_LOCATION["longitude"],
    FOCAL_LOCATION["latitude"],
    marker="*",
    s=220,
    color="#253494",
    edgecolor="white",
    linewidth=1,
    label="Town Center",
    zorder=10,
)

ax.set_title(
    f"Traffic Conditions — {FOCAL_LOCATION['name']}"
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.25)
ax.set_aspect("equal", adjustable="datalim")
ax.legend(title="Traffic Status")

# -------------------------------------------------------------------
# DATE/TIME INFORMATION BOX
# -------------------------------------------------------------------
api_time_text = (
    api_departure_time_utc
    if api_departure_time_utc
    else "automatic = request time"
)

info_text = (
    f"Time mode: {time_mode}\n"
    f"Local time: {requested_departure_local_text}\n"
    f"departureTime for API: {api_time_text}"
)

ax.text(
    0.02,
    0.98,
    info_text,
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox={
        "boxstyle": "round,pad=0.45",
        "facecolor": "white",
        "edgecolor": "0.55",
        "alpha": 0.92,
    },
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print()
print("Local outputs:")
for output_path in [
    route_csv_path,
    segment_csv_path,
    geojson_path,
    figure_path,
]:
    print("-", output_path.resolve())

if error_rows:
    print("-", error_csv_path.resolve())


## Suggested schedule under a 100-request local safety limit

One complete observation cycle uses:

- 8 area entry points
- 2 directions
- 16 API requests

Six complete observation cycles per month use 96 API requests.

Example schedule:

- Monday: 07:00, 12:00, 17:00 WITA
- Saturday: 07:00, 12:00, 17:00 WITA

The local counter only tracks requests made through this notebook/output folder. Always verify actual usage and billing in Google Cloud Console.

### Important limitation for specific dates

For `DRIVE` mode, Google Routes API does **not** allow a `departureTime` in the past. A scheduled time must therefore be in the future. Future traffic is a prediction based on Google's traffic model; it is not a historical traffic archive.
